# 🚀 ReCurRAG — Standard RAG Pipeline

This notebook implements and evaluates the **Retrieval-Augmented Generation (RAG)** pipeline across three distinct dataset types:

| # | Dataset | Type | Source |
|---|---------|------|--------|
| 1 | **arXiv Papers** | Long Documents (Unstructured) | Climate-Finance Research PDFs |
| 2 | **Wine Quality** | Semi-Structured (CSV/Tabular) | UCI ML Repository |
| 3 | **HotpotQA** | Multi-Hop QA (JSON) | EMNLP 2018 Benchmark |

Results are stored in `outputs/rag/` for later comparison with the Recursive Language Model (RLM) approach.

## 📦 Setup & Imports

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import numpy as np

# Suppress tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
from src.rag.pipeline import RAGPipeline
from src.rag.loader import load_documents, chunk_text
from src.rag.embedder import create_embeddings, retrieve, generate_answer

print("✅ All RAG modules imported successfully!")

## 📋 Configuration

In [ ]:
# Load query configuration
with open("configs/queries.json", "r") as f:
    queries_config = json.load(f)

# Dataset paths
DATASET_CONFIGS = {
    "long_docs": {
        "data_path": "data/raw/Long-Docs/papers/",
        "data_type": "long_docs",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "top_k": 5,
    },
    "semi_structured": {
        "data_path": "data/raw/Semi-Structured/wine+quality/",
        "data_type": "semi_structured",
        "chunk_size": 800,
        "chunk_overlap": 150,
        "top_k": 5,
    },
    "multi_hop": {
        "data_path": "data/raw/Multi-HopQA/hotpotqa.json",
        "data_type": "multi_hop",
        "chunk_size": 500,
        "chunk_overlap": 100,
        "top_k": 5,
    },
}

print("📋 Configuration loaded:")
for name, cfg in DATASET_CONFIGS.items():
    print(f"  {name}: {cfg['data_path']}")

---
## 📄 Dataset 1: Long Documents (arXiv Papers)

This dataset tests the RAG pipeline's ability to retrieve and reason over **long-form unstructured text** from academic research papers on Climate Change & Finance.

**Workflow:** `PDF → Text Extraction → Chunking → Embedding → FAISS Retrieval → GPT-4o-mini Generation`

In [ ]:
# Initialize and ingest long documents
rag_long_docs = RAGPipeline(**DATASET_CONFIGS["long_docs"])
rag_long_docs.ingest()

In [ ]:
# Display ingestion statistics
print("\n📊 Long-Docs Ingestion Summary:")
print(f"  Documents loaded: {rag_long_docs.ingest_metadata['num_documents']}")
print(f"  Chunks created:   {rag_long_docs.ingest_metadata['num_chunks']}")
print(f"  Load time:        {rag_long_docs.ingest_metadata['load_time_s']}s")
print(f"  Embed time:       {rag_long_docs.ingest_metadata['embed_time_s']}s")

# Show document sources
print("\n📑 Source Documents:")
for doc in rag_long_docs.documents:
    print(f"  - {doc['source']}: {len(doc['content']):,} characters")

In [ ]:
# Run queries on Long Documents
long_docs_questions = queries_config["long_docs"]["questions"]

print(f"🔍 Running {len(long_docs_questions)} queries on Long-Docs...\n")
long_docs_results = rag_long_docs.run_batch(long_docs_questions)

# Display results
for i, result in enumerate(long_docs_results):
    print(f"\n{'─'*60}")
    print(f"Q{i+1}: {result['question']}")
    print(f"A:  {result['answer'][:300]}..." if len(result['answer']) > 300 else f"A:  {result['answer']}")
    print(f"⏱️  Latency: {result['latency_s']}s | Sources: {result['sources'][:3]}")

In [ ]:
# Save Long-Docs results
long_docs_output_path = rag_long_docs.save_results(long_docs_results)
print(f"\n✅ Long-Docs results saved to: {long_docs_output_path}")

---
## 📊 Dataset 2: Semi-Structured Data (Wine Quality)

This dataset tests the RAG pipeline's ability to perform **tabular reasoning** on structured CSV data from the UCI Machine Learning Repository.

**Challenge:** RAG systems typically struggle with structured data because embedding individual rows loses the schema-level relationships. This is a key area where RLMs are expected to outperform.

**Workflow:** `CSV → Text Conversion → Chunking → Embedding → FAISS Retrieval → GPT-4o-mini Generation`

In [ ]:
# Initialize and ingest semi-structured data
rag_semi = RAGPipeline(**DATASET_CONFIGS["semi_structured"])
rag_semi.ingest()

In [ ]:
# Display ingestion statistics
print("\n📊 Semi-Structured Ingestion Summary:")
print(f"  Documents loaded: {rag_semi.ingest_metadata['num_documents']}")
print(f"  Chunks created:   {rag_semi.ingest_metadata['num_chunks']}")
print(f"  Load time:        {rag_semi.ingest_metadata['load_time_s']}s")
print(f"  Embed time:       {rag_semi.ingest_metadata['embed_time_s']}s")

# Show a preview of the raw data
print("\n📑 Wine Quality Red — Preview:")
df_red = pd.read_csv("data/raw/Semi-Structured/wine+quality/winequality-red.csv", sep=";")
print(f"  Shape: {df_red.shape}")
display(df_red.head())

print("\n📑 Wine Quality White — Preview:")
df_white = pd.read_csv("data/raw/Semi-Structured/wine+quality/winequality-white.csv", sep=";")
print(f"  Shape: {df_white.shape}")
display(df_white.head())

In [ ]:
# Run queries on Semi-Structured data
semi_questions = queries_config["semi_structured"]["questions"]

print(f"🔍 Running {len(semi_questions)} queries on Semi-Structured data...\n")
semi_results = rag_semi.run_batch(semi_questions)

# Display results
for i, result in enumerate(semi_results):
    print(f"\n{'─'*60}")
    print(f"Q{i+1}: {result['question']}")
    print(f"A:  {result['answer'][:300]}..." if len(result['answer']) > 300 else f"A:  {result['answer']}")
    print(f"⏱️  Latency: {result['latency_s']}s")

In [ ]:
# Save Semi-Structured results
semi_output_path = rag_semi.save_results(semi_results)
print(f"\n✅ Semi-Structured results saved to: {semi_output_path}")

---
## 🔗 Dataset 3: Multi-Hop QA (HotpotQA)

This dataset is the **gold standard** for testing multi-step reasoning. HotpotQA questions require combining evidence from multiple documents to arrive at the correct answer.

**Key Difference:** Unlike the other datasets, HotpotQA includes **ground-truth answers** and **supporting facts**, enabling direct accuracy evaluation.

**Expected RAG Limitation:** Standard RAG retrieves top-k chunks based on embedding similarity, but multi-hop questions often require connecting information across non-similar chunks. This is where RLMs with iterative tool use excel.

**Workflow:** `JSON → Context Extraction → Chunking → Embedding → FAISS Retrieval → GPT-4o-mini Generation → Compare with Ground Truth`

In [ ]:
# Initialize and ingest Multi-Hop QA data
rag_multi_hop = RAGPipeline(**DATASET_CONFIGS["multi_hop"])
rag_multi_hop.ingest()

In [ ]:
# Display ingestion statistics
print("\n📊 Multi-Hop QA Ingestion Summary:")
print(f"  Documents loaded: {rag_multi_hop.ingest_metadata['num_documents']}")
print(f"  Chunks created:   {rag_multi_hop.ingest_metadata['num_chunks']}")
print(f"  Load time:        {rag_multi_hop.ingest_metadata['load_time_s']}s")
print(f"  Embed time:       {rag_multi_hop.ingest_metadata['embed_time_s']}s")

# Preview sample questions
print("\n📝 Sample HotpotQA Questions:")
for i, doc in enumerate(rag_multi_hop.documents[:5]):
    print(f"  {i+1}. Q: {doc['question']}")
    print(f"     A: {doc['answer']}")
    print(f"     Level: {doc['level']}, Type: {doc['type']}")
    print()

In [ ]:
# Run HotpotQA evaluation (uses built-in questions + ground truth)
MAX_EVAL_SAMPLES = 50

print(f"🔍 Running HotpotQA evaluation ({MAX_EVAL_SAMPLES} samples)...\n")
multi_hop_results = rag_multi_hop.run_hotpotqa_evaluation(max_samples=MAX_EVAL_SAMPLES)

In [ ]:
# Display Multi-Hop results with ground truth comparison
print("\n📊 Multi-Hop QA Results (first 10):")
print(f"{'─'*80}")

for i, result in enumerate(multi_hop_results[:10]):
    print(f"\nQ{i+1}: {result['question']}")
    print(f"  RAG Answer:    {result['answer'][:200]}")
    print(f"  Ground Truth:  {result['ground_truth_answer']}")
    print(f"  Level: {result['level']} | Latency: {result['latency_s']}s")
    print(f"{'─'*80}")

In [ ]:
# Save Multi-Hop results
multi_hop_output_path = rag_multi_hop.save_results(multi_hop_results)
print(f"\n✅ Multi-Hop QA results saved to: {multi_hop_output_path}")

---
## 📈 RAG Performance Summary

Aggregate statistics across all three datasets. These metrics will be compared against the RLM pipeline in the evaluation stage.

In [ ]:
# Compute summary statistics
summary_data = []

datasets = {
    "Long Documents": long_docs_results,
    "Semi-Structured": semi_results,
    "Multi-Hop QA": multi_hop_results,
}

for name, results in datasets.items():
    latencies = [r["latency_s"] for r in results]
    
    row = {
        "Dataset": name,
        "Num Queries": len(results),
        "Avg Latency (s)": round(np.mean(latencies), 3),
        "Min Latency (s)": round(np.min(latencies), 3),
        "Max Latency (s)": round(np.max(latencies), 3),
        "Std Latency (s)": round(np.std(latencies), 3),
    }
    
    # Add accuracy metrics for Multi-Hop (if answers are available)
    if name == "Multi-Hop QA":
        exact_matches = sum(
            1 for r in results
            if r.get("ground_truth_answer", "").lower().strip()
            in r.get("answer", "").lower().strip()
        )
        row["Exact Match %"] = round(100 * exact_matches / len(results), 1)
    else:
        row["Exact Match %"] = "N/A"
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n📊 RAG Pipeline — Performance Summary")
print("="*70)
display(summary_df)

In [ ]:
# Save combined summary
os.makedirs("outputs/rag", exist_ok=True)

combined_summary = {
    "pipeline": "rag",
    "datasets": {},
}

for ds_name, result_file in [
    ("long_docs", "outputs/rag/long_docs/long_docs_results.json"),
    ("semi_structured", "outputs/rag/semi_structured/semi_structured_results.json"),
    ("multi_hop", "outputs/rag/multi_hop/multi_hop_results.json"),
]:
    if os.path.exists(result_file):
        with open(result_file, "r") as f:
            data = json.load(f)
        combined_summary["datasets"][ds_name] = data["summary"]

with open("outputs/rag/rag_summary.json", "w") as f:
    json.dump(combined_summary, f, indent=2)

print("💾 Combined RAG summary saved to: outputs/rag/rag_summary.json")
print(json.dumps(combined_summary, indent=2))

---
## 🔍 Output Structure

The RAG pipeline results are stored in the following structure:

```
outputs/
├── rag/
│   ├── rag_summary.json              # Combined summary across all datasets
│   ├── long_docs/
│   │   └── long_docs_results.json    # arXiv papers Q&A results
│   ├── semi_structured/
│   │   └── semi_structured_results.json  # Wine Quality Q&A results
│   └── multi_hop/
│       └── multi_hop_results.json    # HotpotQA results + ground truth
└── rlm/                              # (To be populated by RLM pipeline)
    ├── long_docs/
    ├── semi_structured/
    └── multi_hop/
```

Each result JSON file contains:
- `pipeline`: Identifier ("rag" or "rlm")
- `metadata`: Ingestion details (documents, chunks, timing)
- `results`: List of Q&A pairs with context, sources, and latency
- `summary`: Aggregate metrics

### Next Steps
1. Implement the **RLM (Recursive Language Model)** pipeline in `src/rlm/`
2. Run RLM on the same datasets and save to `outputs/rlm/`
3. Build the **Evaluation** module in `src/evaluation/` to compare RAG vs RLM using:
   - Exact Match (EM) & F1 Score
   - Reasoning Depth
   - Context Coverage
   - Latency comparison

In [ ]:
# Verify output files exist
print("📁 Output files generated:")
for root, dirs, files in os.walk("outputs/rag"):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"  {filepath} ({size:,} bytes)")

print("\n✅ RAG pipeline complete! Ready for RLM comparison.")